# LPIPS-direct optimisation control -- believed, not confirmed, to be `tab:app-realism`'s LPIPS control

Writes `exp1_lpips_direct/*`. docs/INVENTORY.md flags this notebook's link to the paper's LPIPS control row as unconfirmed.

**Needs:** DTD, the CNT checkpoint.

**Maintained runnable path:** none. Kept for provenance despite the unconfirmed link.

In [ ]:
# --- Cell 1: mount ---
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = os.environ.get('DATA_ROOT', "/content/drive/MyDrive/DATA_ROOT")  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
dtd_root    = os.path.join(PROJECT_DIR, "dtd/images")
OUT         = os.path.join(PROJECT_DIR, "exp1_lpips_direct"); os.makedirs(OUT, exist_ok=True)


In [ ]:
# --- Cell 2: install + clone CNT ---
!pip install -q pot gdown hydra-core omegaconf einops scikit-learn lpips
%cd /content
![ -d cnt-siga24 ] || git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24
!pip install -q -r cnt-siga24/requirements.in
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!rm -rf /content/cnt-siga24/ckpts && ln -sfn "{os.path.join(PROJECT_DIR,'cnt_ckpts')}" /content/cnt-siga24/ckpts
%cd /content/cnt-siga24


In [ ]:
# --- Cell 3: load model + imports (your exact setup) ---
import torch, numpy as np, matplotlib.pyplot as plt, random, sys, json
from pathlib import Path
sys.path.insert(0, "/content/cnt-siga24")
import torch.nn.functional as F
from torch import nn
import torchvision.models as models
import ot as pot, lpips
from src.inference.core import TexAutoEncoderInference, load_image

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TexAutoEncoderInference(ckpt_path=Path("/content/cnt-siga24/ckpts/finetuned-model256.ckpt"), device=device)
IMG_SIZE = 256
print("CNT loaded on", device)

In [ ]:
# --- Cell 4: CNT helpers (your exact functions) ---
def load_texture(path): return load_image(Path(path), IMG_SIZE).to(device)
def get_generator_inputs(img_tensor):
    with torch.no_grad():
        blob = model.encode(img_tensor); layout = model.splat(blob)
        bf = layout["layer_level_layouts"][-1][0]["blob_features"].clone()
        si = layout["layer_level_layouts"][-1][0]["score_img"].clone()
    return blob, bf, si
def get_feature_map_sizes():
    ratios = model.generator.feature_map_size_ratios
    return [round(IMG_SIZE * r) for r in ratios]
FMS = get_feature_map_sizes()
def build_style_maps(bf, si, feature_map_sizes):
    scores_pyramid = {}; s = si; scores_pyramid[s.shape[-1]] = s
    while s.shape[-1] > min(feature_map_sizes):
        s = F.interpolate(s, s.shape[-1]//2, mode="bilinear", align_corners=False)
        scores_pyramid[s.shape[-1]] = s
    return {size: torch.einsum("bnc,bnhw->bchw", bf, scores_pyramid[size]) for size in set(feature_map_sizes)}
def decode_with_style_maps(blob_ref, style_maps_dict):
    with torch.no_grad():
        layout = model.splat(blob_ref)
        si = layout["layer_level_layouts"][-1][0]["score_img"]; img_size = si.shape[-1]
        ratios = model.generator.feature_map_size_ratios
        fms = [round(img_size*r) for r in ratios]
        sp = {}; s = si; sp[s.shape[-1]] = s
        while s.shape[-1] > min(fms):
            s = F.interpolate(s, s.shape[-1]//2, mode="bilinear", align_corners=False); sp[s.shape[-1]] = s
        feature_in = sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1, model.generator.n_embedding,1,1)
        x = feature_in.clone()
        for i in range(len(fms)):
            x = model.generator.spade_blocks[i](x, style_maps_dict[fms[i]])
            if i == len(fms)-1: break
            elif fms[i] != fms[i+1]: x = F.interpolate(x, size=x.shape[-1]*2, mode="bilinear", align_corners=False)
        return torch.nn.functional.tanh(x.unsqueeze(1)).flatten(1,2)
def tensor_to_image(t):
    x = t.squeeze(0).cpu().clamp(-1,1); return ((x+1)/2).permute(1,2,0).numpy()


In [ ]:
# --- Cell 5: Sinkhorn + linear midpoint (your exact interpolation) ---
def sinkhorn_transport(source_feats, target_feats, eps=0.05, n_iters=100):
    N, M = source_feats.shape[0], target_feats.shape[0]
    C = torch.cdist(source_feats, target_feats).pow(2)
    K = torch.exp(-C/eps)
    a = torch.ones(N,1,device=source_feats.device)/N; b = torch.ones(M,1,device=source_feats.device)/M
    u = torch.ones(N,1,device=source_feats.device)
    for _ in range(n_iters):
        v = b/(K.T@u+1e-10); u = a/(K@v+1e-10)
    plan = u*K*v.T; plan = plan/(plan.sum(1,keepdim=True)+1e-10)
    return plan @ target_feats
def interpolate_sinkhorn_pixel(d, eta, n_subsample=4096):
    sm_a = build_style_maps(d["bf_a"], d["si_a"], FMS); sm_b = build_style_maps(d["bf_b"], d["si_b"], FMS)
    largest = max(FMS); source = sm_a[largest][0]; target = sm_b[largest][0]
    Cc,H,W = source.shape
    src_flat = source.permute(1,2,0).reshape(-1,Cc); tgt_flat = target.permute(1,2,0).reshape(-1,Cc)
    N = src_flat.shape[0]
    idx_s = torch.randperm(N,device=device)[:n_subsample]; idx_t = torch.randperm(N,device=device)[:n_subsample]
    with torch.no_grad(): tt = sinkhorn_transport(src_flat[idx_s], tgt_flat[idx_t])
    nn_idx = torch.cdist(src_flat, src_flat[idx_s]).argmin(1); all_t = tt[nn_idx]
    interp = ((1-eta)*src_flat + eta*all_t).reshape(H,W,Cc).permute(2,0,1).unsqueeze(0)
    sm = {size: (interp if size==largest else F.interpolate(interp,size=size,mode="bilinear",align_corners=False)) for size in set(FMS)}
    return decode_with_style_maps(d["blob_a_gen"], sm)
def interpolate_linear_pixel(d, eta):
    sm_a = build_style_maps(d["bf_a"], d["si_a"], FMS); sm_b = build_style_maps(d["bf_b"], d["si_b"], FMS)
    sm = {size: (1-eta)*sm_a[size] + eta*sm_b[size] for size in set(FMS)}
    return decode_with_style_maps(d["blob_a_gen"], sm)
def build_d(path_a, path_b):
    ia, ib = load_texture(path_a), load_texture(path_b)
    blob_a, bf_a, si_a = get_generator_inputs(ia)
    blob_b, bf_b, si_b = get_generator_inputs(ib)
    return dict(blob_a_gen=blob_a, blob_b_gen=blob_b,
                bf_a=bf_a, si_a=si_a, bf_b=bf_b, si_b=si_b, ia=ia, ib=ib)

In [ ]:
# --- Cell 6: VGG perceptual distance (the metric referred to in the task) ---
class VGGPerceptualDistance:
    def __init__(self, device='cuda'):
        vgg = models.vgg16(pretrained=True).features.eval().to(device)
        for p in vgg.parameters(): p.requires_grad=False
        self.slices=[]; prev=0
        for idx in [3,8,15,22]:
            self.slices.append(torch.nn.Sequential(*list(vgg.children())[prev:idx+1]).eval().to(device)); prev=idx+1
        self.mean=torch.tensor([0.485,0.456,0.406]).view(1,3,1,1).to(device)
        self.std =torch.tensor([0.229,0.224,0.225]).view(1,3,1,1).to(device)
    def extract(self, img):                     # img in [-1,1]
        x=(img+1)/2; x=(x-self.mean)/self.std; feats=[]
        for s in self.slices: x=s(x); feats.append(x)
        return feats
    def distance(self, a, b):                    # DIFFERENTIABLE (no .item())
        return sum((fa-fb).pow(2).sum() for fa,fb in zip(self.extract(a), self.extract(b)))
vgg_dist = VGGPerceptualDistance(device=device)
print("VGG perceptual distance ready")

def midpoint_ratio(mid, ia, ib):
    """Normalized perceptual position of mid between ia and ib (ideal 0.5)."""
    da = vgg_dist.distance(mid, ia).item(); db = vgg_dist.distance(mid, ib).item()
    return da/(da+db+1e-9)

In [ ]:
# --- Cell 7: THE LPIPS/VGG-DIRECT BASELINE (the core request) ---
# Optimize the midpoint IMAGE directly so it is perceptually centered between
# A and B under the SAME VGG perceptual distance used for the ratio. No OT.
def perceptual_direct_midpoint(ia, ib, steps=500, lr=0.05, init=None, eq_w=1.0, prox_w=0.5):
    if init is None: init = 0.5*(ia+ib)                      # init = pixel mean
    x = torch.atanh(init.clamp(-0.999,0.999)).requires_grad_(True)
    opt = torch.optim.Adam([x], lr=lr)
    for _ in range(steps):
        img = torch.tanh(x)
        da = vgg_dist.distance(img, ia); db = vgg_dist.distance(img, ib)
        # perceptually centered: equal distance to both (eq) + minimal total path (prox)
        loss = eq_w*(da-db).pow(2) + prox_w*(da+db)
        opt.zero_grad(); loss.backward(); opt.step()
    return torch.tanh(x).detach()

In [ ]:
def decode_interp(blob_ref, bf_new, si_new):
    with torch.no_grad():
        layout = model.splat(blob_ref)
        layout["layer_level_layouts"][-1][0]["blob_features"] = bf_new
        layout["layer_level_layouts"][-1][0]["score_img"]     = si_new
        return model.decode(layout)

In [ ]:
# --- MISSING CELL: DTD pair definitions (reconstructed from your outputs) ---
import glob, os

# adjust to wherever DTD lives in your Drive
DTD_ROOT = '/content/drive/MyDrive/DATA_ROOT/dtd/images'      # <-- set this to your actual path
assert os.path.isdir(DTD_ROOT), f"DTD not found at {DTD_ROOT}; set DTD_ROOT correctly"

def first_img(category):
    """First image path for a DTD category (deterministic — sorted)."""
    paths = sorted(glob.glob(os.path.join(DTD_ROOT, category, '*.jpg')))
    assert paths, f"no images for DTD category '{category}' under {DTD_ROOT}"
    return paths[0]

# the exact six pairs your run used (from the printed output)
PAIRS = [
    ('dotted',    'honeycombed'),
    ('braided',   'striped'),
    ('marbled',   'freckled'),
    ('chequered', 'scaly'),
    ('banded',    'porous'),
    ('woven',     'cracked'),
]

print(f"{len(PAIRS)} pairs | DTD root: {DTD_ROOT}")
print("sample:", first_img(PAIRS[0][0]))

In [ ]:
# --- Cell 8 (CORRECTED): use YOUR midpoint_ratio definition ---
N_ETA = 21
def your_midpoint_ratio(d, method_fn):
    """EXACT replication of the 7000-pair notebook metric."""
    img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
    img_b = decode_interp(d["blob_a_gen"], d["bf_b"], d["si_b"])   # note: blob_a_gen as ref, as in your pipeline
    total = vgg_dist.distance(img_a, img_b)
    etas = np.linspace(0, 1, N_ETA)
    dnorm = []
    for eta in etas:
        interp = method_fn(d, float(eta))
        dnorm.append(vgg_dist.distance(img_a, interp) / max(total, 1e-10))
    return dnorm[N_ETA // 2], dnorm    # midpoint ratio + full curve

# For the DIRECT baseline there is no method_fn(d,eta) curve — it produces one image.
# To put it on the SAME footing, treat its ratio as dist(A,direct)/dist(A,B):
def direct_ratio(mid, img_a, img_b):
    total = vgg_dist.distance(img_a, img_b)
    return vgg_dist.distance(img_a, mid) / max(total, 1e-10)

rows=[]
for a,b in PAIRS:
    d = build_d(first_img(a), first_img(b))
    img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
    img_b = decode_interp(d["blob_b_gen"], d["bf_b"], d["si_b"])   # B's own blob, matches original
    r_sink,_ = your_midpoint_ratio(d, interpolate_sinkhorn_pixel)
    r_lin,_  = your_midpoint_ratio(d, interpolate_linear_pixel)
    mid_dir  = perceptual_direct_midpoint(d["ia"], d["ib"])
    r_dir    = direct_ratio(mid_dir, img_a, img_b)
    rows.append(dict(pair=f"{a}/{b}", sink=r_sink, lin=r_lin, direct=r_dir))
    print(f"{a}/{b}:  sink {r_sink:.3f}  lin {r_lin:.3f}  direct {r_dir:.3f}")

print("\n=== |ratio - 0.5| (lower = closer to ideal midpoint) ===")
for nm in ['sink','lin','direct']:
    print(f"  {nm:>6}: {np.mean([abs(r[nm]-0.5) for r in rows]):.3f}")

In [ ]:
# --- direct-midpoint baselines (both versions) ---
def tv(x):
    return (x[...,1:,:]-x[...,:-1,:]).abs().mean() + (x[...,:,1:]-x[...,:,:-1]).abs().mean()

def perceptual_direct_midpoint(ia, ib, steps=500, lr=0.05, init=None, eq_w=1.0, prox_w=0.5):
    """UNCONSTRAINED: pure metric optimization. Expect mush."""
    if init is None: init = 0.5*(ia+ib)
    x = torch.atanh(init.clamp(-0.999, 0.999)).requires_grad_(True)
    opt = torch.optim.Adam([x], lr=lr)
    for _ in range(steps):
        img = torch.tanh(x)
        da = vgg_dist.distance(img, ia); db = vgg_dist.distance(img, ib)
        loss = eq_w*(da-db).pow(2) + prox_w*(da+db)
        opt.zero_grad(); loss.backward(); opt.step()
    return torch.tanh(x).detach()

def perceptual_direct_midpoint_fair(d, steps=150, lr=0.01, tv_w=8.0):
    """CONSTRAINED: init from linear midpoint + TV penalty. The fair baseline."""
    init = interpolate_linear_pixel(d, 0.5).detach().clamp(-0.999, 0.999)
    ia, ib = d["ia"], d["ib"]
    x = torch.atanh(init).requires_grad_(True)
    opt = torch.optim.Adam([x], lr=lr)
    for _ in range(steps):
        img = torch.tanh(x)
        da = vgg_dist.distance(img, ia); db = vgg_dist.distance(img, ib)
        loss = (da-db).pow(2) + 0.5*(da+db) + tv_w*tv(img)
        opt.zero_grad(); loss.backward(); opt.step()
    return torch.tanh(x).detach()

print("direct baselines ready")

In [ ]:
# --- CORRECTED COMPARISON: same metric for all methods + manifold axis ---
def as_float(v): return float(v.item()) if torch.is_tensor(v) else float(v)

def balance_ratio(mid, img_a, img_b):
    """da/(da+db). Bounded [0,1] by construction — no method can escape to 4.0.
    0.5 = perfectly perceptually centered. SAME formula for every method."""
    da = as_float(vgg_dist.distance(mid, img_a))
    db = as_float(vgg_dist.distance(mid, img_b))
    return da / (da + db + 1e-9)

# --- axis 2: is it still a real texture? distance to nearest real DTD image in VGG space
import random
random.seed(0)
BANK_CATS = ['dotted','honeycombed','braided','striped','marbled','freckled',
             'chequered','scaly','banded','porous','woven','cracked']
bank = []
for c in BANK_CATS:
    for p in sorted(glob.glob(os.path.join(DTD_ROOT, c, '*.jpg')))[:8]:
        bank.append(load_texture(p))          # <-- your loader -> [-1,1] tensor on device
print(f"manifold bank: {len(bank)} real textures")

def manifold_dist(mid):
    """min VGG distance to any REAL texture. High = off-manifold (mush)."""
    with torch.no_grad():
        return min(as_float(vgg_dist.distance(mid, b)) for b in bank)

rows = []
for a, b in PAIRS:
    d = build_d(first_img(a), first_img(b))
    img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
    img_b = decode_interp(d["blob_b_gen"], d["bf_b"], d["si_b"])

    mids = {
        'sink':   interpolate_sinkhorn_pixel(d, 0.5),
        'lin':    interpolate_linear_pixel(d, 0.5),
        'direct': perceptual_direct_midpoint_fair(d),          # constrained version
        'direct_raw': perceptual_direct_midpoint(d["ia"], d["ib"]),   # unconstrained metric-hacking
    }
    r = {'pair': f"{a}/{b}"}
    for nm, mid in mids.items():
        r[f'bal_{nm}'] = balance_ratio(mid, img_a, img_b)
        r[f'man_{nm}'] = manifold_dist(mid)
    rows.append(r)
    print(f"{a}/{b}")
    for nm in mids:
        print(f"    {nm:11s} balance {r[f'bal_{nm}']:.3f}   manifold-dist {r[f'man_{nm}']:.1f}")

print("\n" + "="*64)
print(f"{'method':12}{'|balance-0.5|':>15}{'manifold dist':>16}   (low/low = good)")
print("="*64)
for nm in ['sink','lin','direct','direct_raw']:
    cb = np.mean([abs(r[f'bal_{nm}'] - 0.5) for r in rows])
    cm = np.mean([r[f'man_{nm}'] for r in rows])
    print(f"{nm:12}{cb:15.3f}{cm:16.1f}")

In [ ]:
print("\n=== |ratio - 0.5| (lower = closer to ideal midpoint) ===")
for nm in ['sink','lin','direct']:
    vals = [float(r[nm].item()) if torch.is_tensor(r[nm]) else float(r[nm]) for r in rows]
    print(f"  {nm:>6}: {np.mean([abs(v-0.5) for v in vals]):.3f}")

In [ ]:
print(f"\n{'method':12}{'|balance-0.5|':>15}{'manifold dist':>16}   (low/low = good)")
print("="*46)
for nm in ['sink','lin','direct','direct_raw']:
    cb = np.mean([abs(r[f'bal_{nm}'] - 0.5) for r in rows])
    cm = np.mean([r[f'man_{nm}'] for r in rows])
    print(f"{nm:12}{cb:15.3f}{cm:16.1f}")

print("\nper-pair:")
for r in rows:
    print(f"  {r['pair']:22}", "  ".join(f"{nm}: bal {r[f'bal_{nm}']:.2f} man {r[f'man_{nm}']:6.0f}"
                                          for nm in ['sink','lin','direct','direct_raw']))

In [ ]:
# --- Cell 9: the decisive visual — does DIRECT look like a texture or mush? ---
fig, axes = plt.subplots(len(PAIRS), 5, figsize=(15, 3*len(PAIRS)))
for i,(a,b) in enumerate(PAIRS):
    d = build_d(first_img(a), first_img(b)); ia,ib=d["ia"],d["ib"]
    cols = [('A',ia), ('Sinkhorn',interpolate_sinkhorn_pixel(d,0.5)),
            ('linear',interpolate_linear_pixel(d,0.5)),
            ('VGG-direct',perceptual_direct_midpoint(ia,ib)), ('B',ib)]
    for j,(nm,t) in enumerate(cols):
        axes[i,j].imshow(np.clip(tensor_to_image(t),0,1)); axes[i,j].axis('off')
        if i==0: axes[i,j].set_title(nm)
plt.tight_layout(); plt.savefig(f"{OUT}/lpips_direct_vs_sinkhorn.png",dpi=130,bbox_inches='tight'); plt.show()
print("If VGG-direct is mush while Sinkhorn is textural -> the metric alone is insufficient. The point, demonstrated.")

In [ ]:
# --- Constrained VGG-direct baseline (fair version) ---
# Forces the direct-optimized midpoint to stay image-like:
#   - init from the LINEAR midpoint (a real texture, not noise)
#   - TV penalty for smoothness (kills adversarial static)
#   - fewer steps / smaller LR (can't wander far into noise)
def tv(x):
    return (x[...,1:,:]-x[...,:-1,:]).abs().mean() + (x[...,:,1:]-x[...,:,:-1]).abs().mean()

def perceptual_direct_midpoint_fair(d, steps=150, lr=0.01, tv_w=8.0):
    # start from the linear pixel midpoint (already a plausible texture, in [-1,1])
    init = interpolate_linear_pixel(d, 0.5).detach().clamp(-0.999, 0.999)
    ia, ib = d["ia"], d["ib"]
    x = torch.atanh(init).requires_grad_(True)
    opt = torch.optim.Adam([x], lr=lr)
    for _ in range(steps):
        img = torch.tanh(x)
        da = vgg_dist.distance(img, ia)
        db = vgg_dist.distance(img, ib)
        # perceptually centered (equidistant) + minimal path + stay smooth/plausible
        loss = (da - db).pow(2) + 0.5*(da + db) + tv_w*tv(img)
        opt.zero_grad(); loss.backward(); opt.step()
    return torch.tanh(x).detach()

# --- re-run the comparison with the FAIR direct baseline ---
def as_float(v): return float(v.item()) if torch.is_tensor(v) else float(v)

rows=[]
for a,b in PAIRS:
    d = build_d(first_img(a), first_img(b))
    img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
    img_b = decode_interp(d["blob_b_gen"], d["bf_b"], d["si_b"])
    r_sink,_ = your_midpoint_ratio(d, interpolate_sinkhorn_pixel)
    r_lin,_  = your_midpoint_ratio(d, interpolate_linear_pixel)
    mid_dir  = perceptual_direct_midpoint_fair(d)
    r_dir    = as_float(direct_ratio(mid_dir, img_a, img_b))
    rows.append(dict(pair=f"{a}/{b}", sink=as_float(r_sink), lin=as_float(r_lin), direct=r_dir))
    print(f"{a}/{b}:  sink {rows[-1]['sink']:.3f}  lin {rows[-1]['lin']:.3f}  direct(fair) {r_dir:.3f}")

print("\n=== |ratio - 0.5| (lower = closer to ideal midpoint) ===")
for nm in ['sink','lin','direct']:
    print(f"  {nm:>6}: {np.mean([abs(r[nm]-0.5) for r in rows]):.3f}")

# --- visual: is the FAIR direct midpoint textural now, and how does it compare? ---
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(PAIRS), 5, figsize=(15, 3*len(PAIRS)))
for i,(a,b) in enumerate(PAIRS):
    d = build_d(first_img(a), first_img(b))
    cols = [('A', d["ia"]),
            ('Sinkhorn', interpolate_sinkhorn_pixel(d,0.5)),
            ('linear', interpolate_linear_pixel(d,0.5)),
            ('direct(fair)', perceptual_direct_midpoint_fair(d)),
            ('B', d["ib"])]
    for j,(nm,t) in enumerate(cols):
        axes[i,j].imshow(np.clip(tensor_to_image(t),0,1)); axes[i,j].axis('off')
        if i==0: axes[i,j].set_title(nm)
plt.tight_layout(); plt.savefig(f"{OUT}/direct_fair_vs_sinkhorn.png",dpi=130,bbox_inches='tight'); plt.show()

In [ ]:
import random, pickle, time, os
import numpy as np

# ---------- config ----------
N_PAIRS   = 50                      # bump to 200 after sanity check
SEED      = 0
CKPT      = '/content/drive/MyDrive/DATA_ROOT/direct_baseline_ckpt.pkl'
METHODS   = ['sink', 'lin', 'direct', 'direct_raw']

# ---------- pair sampling: random across ALL DTD categories, reproducible ----------
ALL_CATS = sorted([d for d in os.listdir(DTD_ROOT) if os.path.isdir(os.path.join(DTD_ROOT, d))])
rng = random.Random(SEED)

def sample_pairs(n):
    pairs = []
    while len(pairs) < n:
        ca, cb = rng.sample(ALL_CATS, 2)                 # cross-category (the hard case)
        pa = sorted(glob.glob(os.path.join(DTD_ROOT, ca, '*.jpg')))
        pb = sorted(glob.glob(os.path.join(DTD_ROOT, cb, '*.jpg')))
        if pa and pb:
            pairs.append((rng.choice(pa), rng.choice(pb), ca, cb))
    return pairs

PAIRS_BIG = sample_pairs(N_PAIRS)
print(f"{len(PAIRS_BIG)} random cross-category pairs from {len(ALL_CATS)} DTD categories")

# ---------- manifold bank (real textures, broad sample) ----------
bank = []
for c in rng.sample(ALL_CATS, min(24, len(ALL_CATS))):
    for p in sorted(glob.glob(os.path.join(DTD_ROOT, c, '*.jpg')))[:4]:
        bank.append(load_texture(p))
print(f"manifold bank: {len(bank)} real textures")

def as_float(v): return float(v.item()) if torch.is_tensor(v) else float(v)

def balance_ratio(mid, ia, ib):
    da = as_float(vgg_dist.distance(mid, ia)); db = as_float(vgg_dist.distance(mid, ib))
    return da / (da + db + 1e-9)

def manifold_dist_norm(mid, ia, ib, d_ab):
    """min distance to ANY real texture (sources + bank), NORMALIZED by dist(A,B).
    ~0 = sits on a real texture.  ~1 = as far from real as A is from B."""
    with torch.no_grad():
        m = min(as_float(vgg_dist.distance(mid, r)) for r in [ia, ib] + bank)
    return m / max(d_ab, 1e-9)

# ---------- resume ----------
if os.path.exists(CKPT):
    rows = pickle.load(open(CKPT, 'rb')); start = len(rows)
    print(f"resuming from pair {start}/{N_PAIRS}")
else:
    rows, start = [], 0

t0 = time.time()
for n in range(start, N_PAIRS):
    pa, pb, ca, cb = PAIRS_BIG[n]
    try:
        d = build_d(pa, pb)
        img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
        img_b = decode_interp(d["blob_b_gen"], d["bf_b"], d["si_b"])
        d_ab  = as_float(vgg_dist.distance(img_a, img_b))

        mids = {
            'sink':       interpolate_sinkhorn_pixel(d, 0.5),
            'lin':        interpolate_linear_pixel(d, 0.5),
            'direct':     perceptual_direct_midpoint_fair(d),
            'direct_raw': perceptual_direct_midpoint(d["ia"], d["ib"]),
        }
        r = {'pair': f"{ca}/{cb}"}
        for nm, mid in mids.items():
            r[f'bal_{nm}'] = balance_ratio(mid, img_a, img_b)
            r[f'man_{nm}'] = manifold_dist_norm(mid, img_a, img_b, d_ab)
        rows.append(r)
    except Exception as e:
        print(f"  skip {ca}/{cb}: {type(e).__name__}")
        continue

    if n % 10 == 0 and n > start:
        pickle.dump(rows, open(CKPT, 'wb'))
        el = time.time() - t0
        print(f"  {n}/{N_PAIRS}  {el/60:.1f}m  eta {el/(n-start+1)*(N_PAIRS-n-1)/60:.1f}m  [ckpt]")

pickle.dump(rows, open(CKPT, 'wb'))
print(f"\ndone: {len(rows)} pairs in {(time.time()-t0)/60:.1f} min\n")

# ---------- results ----------
print(f"{'method':13}{'|balance-0.5|':>15}{'manifold (norm)':>18}   (low = centered / low = real)")
print("=" * 62)
for nm in METHODS:
    cb_ = np.mean([abs(r[f'bal_{nm}'] - 0.5) for r in rows])
    cm_ = np.mean([r[f'man_{nm}'] for r in rows])
    sb  = np.std([abs(r[f'bal_{nm}'] - 0.5) for r in rows])
    sm  = np.std([r[f'man_{nm}'] for r in rows])
    print(f"{nm:13}{cb_:9.3f} ±{sb:.3f}{cm_:12.3f} ±{sm:.3f}")

print(f"\nwin rates vs sinkhorn (n={len(rows)}):")
for nm in ['lin','direct','direct_raw']:
    wb = np.mean([abs(r[f'bal_{nm}']-0.5) < abs(r['bal_sink']-0.5) for r in rows])
    wm = np.mean([r[f'man_{nm}'] < r['man_sink'] for r in rows])
    print(f"  {nm:11} beats sinkhorn on centering {wb*100:4.0f}%   on manifold {wm*100:4.0f}%")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

colors = {'sink':'tab:green','lin':'tab:blue','direct':'tab:orange','direct_raw':'tab:red'}
labels = {'sink':'Sinkhorn OT','lin':'pixel linear','direct':'VGG-direct (constrained)','direct_raw':'VGG-direct (raw)'}

for nm in METHODS:
    x = [abs(r[f'bal_{nm}']-0.5) for r in rows]
    y = [r[f'man_{nm}'] for r in rows]
    ax[0].scatter(x, y, s=18, alpha=0.45, c=colors[nm], label=labels[nm])
    ax[1].scatter(np.mean(x), np.mean(y), s=260, c=colors[nm], edgecolors='k', zorder=3, label=labels[nm])
    ax[1].errorbar(np.mean(x), np.mean(y), xerr=np.std(x)/np.sqrt(len(x)),
                   yerr=np.std(y)/np.sqrt(len(y)), c='k', zorder=2)

for a, t in zip(ax, [f'per-pair (n={len(rows)})', 'means ± s.e.']):
    a.set_xlabel('|balance − 0.5|     (0 = perfectly centered)')
    a.set_ylabel('manifold distance / dist(A,B)   (0 = a real texture)')
    a.set_title(t); a.legend(fontsize=8); a.grid(alpha=0.25)
ax[1].annotate('GOOD\n(centered AND real)', xy=(0.02, 0.02), xycoords='axes fraction',
               fontsize=9, color='green', va='bottom')
plt.suptitle('Direct perceptual optimization is more "centered" but leaves the texture manifold', fontsize=11)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/DATA_ROOT/direct_two_axis.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
show = rows[:0] or PAIRS_BIG[:4]      # first 4 pairs
fig, axes = plt.subplots(len(show), 6, figsize=(17, 3*len(show)))
for i,(pa, pb, ca, cb) in enumerate(show):
    d = build_d(pa, pb)
    cols = [('A: '+ca, d["ia"]),
            ('Sinkhorn', interpolate_sinkhorn_pixel(d, 0.5)),
            ('linear', interpolate_linear_pixel(d, 0.5)),
            ('direct (fair)', perceptual_direct_midpoint_fair(d)),
            ('direct (raw)', perceptual_direct_midpoint(d["ia"], d["ib"])),
            ('B: '+cb, d["ib"])]
    for j,(nm,t) in enumerate(cols):
        axes[i,j].imshow(np.clip(tensor_to_image(t),0,1)); axes[i,j].axis('off')
        if i==0: axes[i,j].set_title(nm, fontsize=10)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/DATA_ROOT/direct_qualitative.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
N_PAIRS = 200
CKPT    = '/content/drive/MyDrive/DATA_ROOT/direct_baseline_200_ckpt.pkl'
rng = random.Random(1)   # new seed so it's a fresh 200, not the same 50

def sample_split_pairs(n, kind):
    """kind='within' -> same category, different image; 'cross' -> different categories"""
    pairs = []
    while len(pairs) < n:
        if kind == 'within':
            c = rng.choice(ALL_CATS)
            ps = sorted(glob.glob(os.path.join(DTD_ROOT, c, '*.jpg')))
            if len(ps) >= 2:
                a, b = rng.sample(ps, 2); pairs.append((a, b, c, c))
        else:
            ca, cb = rng.sample(ALL_CATS, 2)
            pa = sorted(glob.glob(os.path.join(DTD_ROOT, ca, '*.jpg')))
            pb = sorted(glob.glob(os.path.join(DTD_ROOT, cb, '*.jpg')))
            if pa and pb: pairs.append((rng.choice(pa), rng.choice(pb), ca, cb))
    return pairs

# half within-category, half cross-category
PAIRS_BIG = ([(*p, 'within') for p in sample_split_pairs(N_PAIRS//2, 'within')] +
             [(*p, 'cross')  for p in sample_split_pairs(N_PAIRS//2, 'cross')])
rng.shuffle(PAIRS_BIG)
print(f"{len(PAIRS_BIG)} pairs: {N_PAIRS//2} within + {N_PAIRS//2} cross")

if os.path.exists(CKPT):
    rows = pickle.load(open(CKPT,'rb')); start = len(rows)
    print(f"resuming {start}/{N_PAIRS}")
else:
    rows, start = [], 0

t0 = time.time()
for n in range(start, len(PAIRS_BIG)):
    pa, pb, ca, cb, split = PAIRS_BIG[n]
    try:
        d = build_d(pa, pb)
        img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
        img_b = decode_interp(d["blob_b_gen"], d["bf_b"], d["si_b"])
        d_ab  = as_float(vgg_dist.distance(img_a, img_b))
        mids = {'sink':interpolate_sinkhorn_pixel(d,0.5), 'lin':interpolate_linear_pixel(d,0.5),
                'direct':perceptual_direct_midpoint_fair(d), 'direct_raw':perceptual_direct_midpoint(d["ia"],d["ib"])}
        r = {'pair':f"{ca}/{cb}", 'split':split}
        for nm,mid in mids.items():
            r[f'bal_{nm}'] = balance_ratio(mid, img_a, img_b)
            r[f'man_{nm}'] = manifold_dist_norm(mid, img_a, img_b, d_ab)
        rows.append(r)
    except Exception as e:
        print(f"  skip {ca}/{cb}: {type(e).__name__}"); continue
    if n % 10 == 0 and n > start:
        pickle.dump(rows, open(CKPT,'wb'))
        el=time.time()-t0; print(f"  {n}/{len(PAIRS_BIG)}  {el/60:.1f}m  eta {el/(n-start+1)*(len(PAIRS_BIG)-n-1)/60:.1f}m")
pickle.dump(rows, open(CKPT,'wb'))
print(f"\ndone {len(rows)} pairs in {(time.time()-t0)/60:.1f}m")

# ---- split-aware table (drops into your PTD slide as a 4th column) ----
for split in ['within','cross']:
    sub = [r for r in rows if r['split']==split]
    print(f"\n=== {split.upper()} (n={len(sub)}) — |balance-0.5| / manifold ===")
    for nm in ['sink','lin','direct','direct_raw']:
        cb_=np.mean([abs(r[f'bal_{nm}']-0.5) for r in sub]); cm_=np.mean([r[f'man_{nm}'] for r in sub])
        print(f"  {nm:11} centering {cb_:.3f}   manifold {cm_:.3f}")

In [ ]:
# ============================================================
# DIAGNOSTIC: why does balance_ratio disagree with the PTD slide?
# Compute BOTH metrics on the same pairs, same midpoints.
# ============================================================
diag = []
for n in range(min(60, len(PAIRS_BIG))):
    pa, pb, ca, cb, split = PAIRS_BIG[n]
    try:
        d = build_d(pa, pb)
        img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
        img_b = decode_interp(d["blob_b_gen"], d["bf_b"], d["si_b"])
        d_ab  = as_float(vgg_dist.distance(img_a, img_b))

        r = {'pair': f"{ca}/{cb}", 'split': split}
        for nm, fn in [('sink', interpolate_sinkhorn_pixel), ('lin', interpolate_linear_pixel)]:
            mid = fn(d, 0.5)
            # metric 1: today's symmetric balance
            r[f'bal_{nm}'] = balance_ratio(mid, img_a, img_b)
            # metric 2: THE PTD-SLIDE METRIC — distance from A, normalized by total
            r[f'ratio_{nm}'] = as_float(vgg_dist.distance(img_a, mid)) / max(d_ab, 1e-9)
        diag.append(r)
    except Exception as e:
        continue

print(f"n={len(diag)}\n")
print(f"{'metric':28}{'sink':>10}{'linear':>10}{'sink wins':>12}")
print("="*60)
for key, label in [('bal', '|balance - 0.5|'), ('ratio', '|ratio - 0.5|  (PTD slide)')]:
    s = np.mean([abs(r[f'{key}_sink'] - 0.5) for r in diag])
    l = np.mean([abs(r[f'{key}_lin']  - 0.5) for r in diag])
    w = np.mean([abs(r[f'{key}_sink']-0.5) < abs(r[f'{key}_lin']-0.5) for r in diag])
    print(f"{label:28}{s:10.3f}{l:10.3f}{w*100:11.0f}%")

print(f"\nraw (unsigned) values — is one method systematically overshooting?")
for key in ['bal','ratio']:
    s = np.mean([r[f'{key}_sink'] for r in diag]); l = np.mean([r[f'{key}_lin'] for r in diag])
    print(f"  {key:6} sink {s:.3f}   linear {l:.3f}   (0.5 = ideal)")

In [ ]:
# ============================================================
# FINAL TABLE — PTD-slide metric (ratio) + manifold, on the n=200 set
# ratio = dist(A, mid) / dist(A, B). Ideal = 0.5.
# Recomputed from scratch so all 4 methods use the SAME correct metric.
# ============================================================
CKPT2 = '/content/drive/MyDrive/DATA_ROOT/direct_final_ckpt.pkl'
if os.path.exists(CKPT2):
    rows2 = pickle.load(open(CKPT2,'rb')); start2 = len(rows2)
    print(f"resuming {start2}/{len(PAIRS_BIG)}")
else:
    rows2, start2 = [], 0

t0 = time.time()
for n in range(start2, len(PAIRS_BIG)):
    pa, pb, ca, cb, split = PAIRS_BIG[n]
    try:
        d = build_d(pa, pb)
        img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
        img_b = decode_interp(d["blob_b_gen"], d["bf_b"], d["si_b"])
        d_ab  = as_float(vgg_dist.distance(img_a, img_b))

        mids = {'sink':       interpolate_sinkhorn_pixel(d, 0.5),
                'lin':        interpolate_linear_pixel(d, 0.5),
                'direct':     perceptual_direct_midpoint_fair(d),
                'direct_raw': perceptual_direct_midpoint(d["ia"], d["ib"])}
        r = {'pair': f"{ca}/{cb}", 'split': split}
        for nm, mid in mids.items():
            r[f'ratio_{nm}'] = as_float(vgg_dist.distance(img_a, mid)) / max(d_ab, 1e-9)
            r[f'man_{nm}']   = manifold_dist_norm(mid, img_a, img_b, d_ab)
        rows2.append(r)
    except Exception:
        continue
    if n % 20 == 0 and n > start2:
        pickle.dump(rows2, open(CKPT2,'wb'))
        el = time.time()-t0
        print(f"  {n}/{len(PAIRS_BIG)}  {el/60:.1f}m  eta {el/(n-start2+1)*(len(PAIRS_BIG)-n-1)/60:.1f}m")
pickle.dump(rows2, open(CKPT2,'wb'))
print(f"\ndone in {(time.time()-t0)/60:.1f}m\n")

METHODS = ['sink','lin','direct','direct_raw']
NICE = {'sink':'Sinkhorn OT','lin':'pixel linear','direct':'VGG-direct (constr.)','direct_raw':'VGG-direct (raw)'}

for split in ['within','cross','ALL']:
    sub = rows2 if split=='ALL' else [r for r in rows2 if r['split']==split]
    print(f"=== {split.upper()} (n={len(sub)}) ===")
    print(f"{'method':22}{'ratio':>8}{'|ratio-0.5|':>13}{'manifold':>11}")
    print("-"*54)
    for nm in METHODS:
        raw = np.mean([r[f'ratio_{nm}'] for r in sub])
        cen = np.mean([abs(r[f'ratio_{nm}']-0.5) for r in sub])
        man = np.mean([r[f'man_{nm}'] for r in sub])
        print(f"{NICE[nm]:22}{raw:8.3f}{cen:13.3f}{man:11.3f}")
    print()

print(f"Sinkhorn win-rate vs each baseline (n={len(rows2)}):")
for nm in ['lin','direct','direct_raw']:
    wc = np.mean([abs(r['ratio_sink']-0.5) < abs(r[f'ratio_{nm}']-0.5) for r in rows2])
    wm = np.mean([r['man_sink'] < r[f'man_{nm}'] for r in rows2])
    print(f"  vs {NICE[nm]:22} centering {wc*100:3.0f}%   manifold {wm*100:3.0f}%")

In [ ]:
# ============================================================
# FAIR DIRECT BASELINE — optimize the EXACT metric we grade on.
#   objective: (dist(A,mid)/dist(A,B) - 0.5)^2   [+ optional smoothness prior]
# If it STILL lands off-manifold, the result is un-attackable.
# ============================================================

def direct_ratio_opt(d, img_a, img_b, d_ab, steps=400, lr=0.03,
                     init_from_linear=True, tv_w=0.0, prox_w=0.0):
    """Directly optimize ratio -> 0.5. This is the strongest form of the baseline."""
    if init_from_linear:
        init = interpolate_linear_pixel(d, 0.5).detach().clamp(-0.999, 0.999)
    else:
        init = (0.5*(d["ia"] + d["ib"])).clamp(-0.999, 0.999)
    x = torch.atanh(init).requires_grad_(True)
    opt = torch.optim.Adam([x], lr=lr)
    for _ in range(steps):
        img = torch.tanh(x)
        da = vgg_dist.distance(img, img_a)
        db = vgg_dist.distance(img, img_b)
        ratio = da / (d_ab + 1e-9)
        loss = (ratio - 0.5).pow(2)                 # THE graded metric, optimized directly
        if prox_w: loss = loss + prox_w*(da + db)/(d_ab + 1e-9)
        if tv_w:   loss = loss + tv_w*tv(img)
        opt.zero_grad(); loss.backward(); opt.step()
    return torch.tanh(x).detach()

CKPT3 = '/content/drive/MyDrive/DATA_ROOT/direct_fair_objective_ckpt.pkl'
N_FAIR = 100                                    # 100 is plenty to settle this

if os.path.exists(CKPT3):
    rows3 = pickle.load(open(CKPT3,'rb')); start3 = len(rows3)
    print(f"resuming {start3}/{N_FAIR}")
else:
    rows3, start3 = [], 0

t0 = time.time()
for n in range(start3, N_FAIR):
    pa, pb, ca, cb, split = PAIRS_BIG[n]
    try:
        d = build_d(pa, pb)
        img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
        img_b = decode_interp(d["blob_b_gen"], d["bf_b"], d["si_b"])
        d_ab  = as_float(vgg_dist.distance(img_a, img_b))

        mids = {
            'sink':        interpolate_sinkhorn_pixel(d, 0.5),
            'lin':         interpolate_linear_pixel(d, 0.5),
            # NEW: optimizes the graded metric directly
            'ratio_free':  direct_ratio_opt(d, img_a, img_b, d_ab, init_from_linear=False),
            'ratio_prior': direct_ratio_opt(d, img_a, img_b, d_ab, init_from_linear=True,
                                            tv_w=4.0, prox_w=0.1),
        }
        r = {'pair': f"{ca}/{cb}", 'split': split}
        for nm, mid in mids.items():
            r[f'ratio_{nm}'] = as_float(vgg_dist.distance(img_a, mid)) / max(d_ab, 1e-9)
            r[f'man_{nm}']   = manifold_dist_norm(mid, img_a, img_b, d_ab)
        rows3.append(r)
    except Exception as e:
        print(f"  skip {ca}/{cb}: {type(e).__name__}"); continue

    if n % 10 == 0 and n > start3:
        pickle.dump(rows3, open(CKPT3,'wb'))
        el = time.time()-t0
        print(f"  {n}/{N_FAIR}  {el/60:.1f}m  eta {el/(n-start3+1)*(N_FAIR-n-1)/60:.1f}m")

pickle.dump(rows3, open(CKPT3,'wb'))
print(f"\ndone in {(time.time()-t0)/60:.1f}m\n")

M3 = ['sink','lin','ratio_free','ratio_prior']
N3 = {'sink':'Sinkhorn OT', 'lin':'pixel linear',
      'ratio_free':'direct (metric-optimal, free)', 'ratio_prior':'direct (metric-optimal + prior)'}

print(f"{'method':34}{'ratio':>8}{'|ratio-0.5|':>13}{'manifold':>11}   (n={len(rows3)})")
print("="*70)
for nm in M3:
    raw = np.mean([r[f'ratio_{nm}'] for r in rows3])
    cen = np.mean([abs(r[f'ratio_{nm}']-0.5) for r in rows3])
    man = np.mean([r[f'man_{nm}'] for r in rows3])
    print(f"{N3[nm]:34}{raw:8.3f}{cen:13.3f}{man:11.3f}")

print(f"\nSinkhorn win-rate:")
for nm in ['lin','ratio_free','ratio_prior']:
    wc = np.mean([abs(r['ratio_sink']-0.5) < abs(r[f'ratio_{nm}']-0.5) for r in rows3])
    wm = np.mean([r['man_sink'] < r[f'man_{nm}'] for r in rows3])
    print(f"  vs {N3[nm]:32} centering {wc*100:3.0f}%   manifold {wm*100:3.0f}%")

print("\nKEY: if direct now WINS centering but LOSES manifold -> the dissociation result.")
print("     if direct loses BOTH -> even the metric-optimal baseline can't reach the midpoint.")

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(17, 9))
for i in range(3):
    pa, pb, ca, cb, _ = PAIRS_BIG[i]
    d = build_d(pa, pb)
    img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
    img_b = decode_interp(d["blob_b_gen"], d["bf_b"], d["si_b"])
    d_ab  = as_float(vgg_dist.distance(img_a, img_b))
    cols = [(f'A: {ca}', d["ia"]),
            ('Sinkhorn', interpolate_sinkhorn_pixel(d, 0.5)),
            ('linear', interpolate_linear_pixel(d, 0.5)),
            ('direct: metric-optimal', direct_ratio_opt(d, img_a, img_b, d_ab, init_from_linear=False)),
            ('direct: + TV prior', direct_ratio_opt(d, img_a, img_b, d_ab, init_from_linear=True, tv_w=4.0, prox_w=0.1)),
            (f'B: {cb}', d["ib"])]
    for j,(nm,t) in enumerate(cols):
        axes[i,j].imshow(np.clip(tensor_to_image(t),0,1)); axes[i,j].axis('off')
        if i==0: axes[i,j].set_title(nm, fontsize=9)
        if j in (3,4):
            rr = as_float(vgg_dist.distance(img_a, t))/max(d_ab,1e-9)
            axes[i,j].set_xlabel(f"ratio={rr:.2f}", fontsize=8); axes[i,j].axis('on')
            axes[i,j].set_xticks([]); axes[i,j].set_yticks([])
plt.suptitle('Even when it optimizes the exact metric, direct optimization does not produce a texture', fontsize=11)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/DATA_ROOT/direct_metric_optimal.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
!rm -rf ~/.cache/huggingface/hub/models--timm--vit_small_patch14_dinov2.lvd142m

In [ ]:
import timm, torch
dino = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True, num_classes=0).eval().to(device)
for p in dino.parameters(): p.requires_grad_(False)
print("dino loaded")

In [ ]:
# ============================================================
# BENCHMARK SUITE v2 — metrics that direct optimization cannot game
# (no downloads: Gram reuses the loaded VGG, HF is a conv kernel)
# ============================================================

# --- Gram (classical texture statistic; blur destroys it) ---
def gram_feats(img):
    """Gram matrices from VGG layers — texture statistics, not perceptual distance."""
    fs = vgg_dist.extract(img)
    out = []
    for f in fs:
        b, c, h, w = f.shape
        x = f.reshape(c, h*w)
        out.append((x @ x.t()) / (c*h*w))
    return out

def gram_dist(a, b):
    with torch.no_grad():
        return sum((ga-gb).pow(2).sum().item() for ga, gb in zip(gram_feats(a), gram_feats(b)))

def gram_midpoint_ratio(mid, img_a, img_b, dab_gram):
    """Is the midpoint's TEXTURE STATISTIC halfway between A and B? ideal 0.5"""
    return gram_dist(mid, img_a) / max(dab_gram, 1e-12)

def gram_manifold(mid, img_a, img_b, dab_gram):
    """Gram distance to nearest real texture, normalized. Blur -> high. ideal ~0"""
    return min(gram_dist(mid, r) for r in [img_a, img_b] + bank[:24]) / max(dab_gram, 1e-12)

# --- sharpness: catches BOTH blur (too low) and adversarial static (too high) ---
_LAP = torch.tensor([[0,1,0],[1,-4,1],[0,1,0]], dtype=torch.float32, device=device).view(1,1,3,3)

def hf_energy(img):
    """high-frequency energy via Laplacian response"""
    with torch.no_grad():
        g = img.mean(1, keepdim=True)
        return F.conv2d(g, _LAP, padding=1).pow(2).mean().item()

def hf_deviation(mid, img_a, img_b):
    """|log(HF_mid / HF_expected)|, expected = geometric mean of A,B. 0 = natural sharpness.
    blur -> large (too little HF).  adversarial static -> large (too much HF)."""
    ha, hb, hm = hf_energy(img_a), hf_energy(img_b), hf_energy(mid)
    expect = np.sqrt(max(ha, 1e-9) * max(hb, 1e-9))
    return abs(np.log(max(hm, 1e-9) / expect))

def hf_signed(mid, img_a, img_b):
    """signed version: NEGATIVE = blurrier than expected, POSITIVE = noisier."""
    ha, hb, hm = hf_energy(img_a), hf_energy(img_b), hf_energy(mid)
    expect = np.sqrt(max(ha, 1e-9) * max(hb, 1e-9))
    return np.log(max(hm, 1e-9) / expect)

print("benchmark suite v2 ready: gram ratio / gram manifold / HF deviation (+ signed HF)")

In [ ]:
CKPT4 = '/content/drive/MyDrive/DATA_ROOT/benchmark_v2_ckpt.pkl'
N4 = 100
if os.path.exists(CKPT4):
    rows4 = pickle.load(open(CKPT4,'rb')); start4 = len(rows4)
    print(f"resuming {start4}/{N4}")
else:
    rows4, start4 = [], 0

t0 = time.time()
for n in range(start4, N4):
    pa, pb, ca, cb, split = PAIRS_BIG[n]
    try:
        d = build_d(pa, pb)
        img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
        img_b = decode_interp(d["blob_b_gen"], d["bf_b"], d["si_b"])
        d_ab      = as_float(vgg_dist.distance(img_a, img_b))
        d_ab_gram = gram_dist(img_a, img_b)

        mids = {
            'sink':        interpolate_sinkhorn_pixel(d, 0.5),
            'lin':         interpolate_linear_pixel(d, 0.5),
            'ratio_free':  direct_ratio_opt(d, img_a, img_b, d_ab, init_from_linear=False),
            'ratio_prior': direct_ratio_opt(d, img_a, img_b, d_ab, init_from_linear=True, tv_w=4.0, prox_w=0.1),
        }
        r = {'pair': f"{ca}/{cb}"}
        for nm, mid in mids.items():
            r[f'vggratio_{nm}']  = as_float(vgg_dist.distance(img_a, mid)) / max(d_ab, 1e-9)
            r[f'gramratio_{nm}'] = gram_midpoint_ratio(mid, img_a, img_b, d_ab_gram)
            r[f'grammani_{nm}']  = gram_manifold(mid, img_a, img_b, d_ab_gram)
            r[f'hf_{nm}']        = hf_deviation(mid, img_a, img_b)
            r[f'hfs_{nm}']       = hf_signed(mid, img_a, img_b)
        rows4.append(r)
    except Exception as e:
        print(f"  skip {ca}/{cb}: {type(e).__name__}"); continue

    if n % 10 == 0 and n > start4:
        pickle.dump(rows4, open(CKPT4,'wb'))
        el = time.time()-t0
        print(f"  {n}/{N4}  {el/60:.1f}m  eta {el/(n-start4+1)*(N4-n-1)/60:.1f}m")

pickle.dump(rows4, open(CKPT4,'wb'))
print(f"\ndone {(time.time()-t0)/60:.1f}m — n={len(rows4)}\n")

M  = ['sink','lin','ratio_free','ratio_prior']
NM = {'sink':'Sinkhorn OT','lin':'pixel linear','ratio_free':'direct (metric-opt)','ratio_prior':'direct (+TV prior)'}
METRICS = [
    ('vggratio',  '|VGG ratio - 0.5|',  lambda v: abs(v-0.5), 'VGG-based: GAMEABLE'),
    ('gramratio', '|Gram ratio - 0.5|', lambda v: abs(v-0.5), 'texture statistic'),
    ('grammani',  'Gram manifold',      lambda v: v,          'blur -> high'),
    ('hf',        'HF deviation',       lambda v: v,          '0 = natural sharpness'),
]
for key, label, fn, note in METRICS:
    vals = {nm: np.mean([fn(r[f'{key}_{nm}']) for r in rows4]) for nm in M}
    best = min(vals, key=vals.get)
    print(f"{label:22} ({note})")
    for nm in M:
        print(f"    {NM[nm]:22}{vals[nm]:10.3f}{'   <-- BEST' if nm==best else ''}")
    print()

print("signed HF (NEGATIVE = blurry, POSITIVE = adversarial static, 0 = natural):")
for nm in M:
    print(f"    {NM[nm]:22}{np.mean([r[f'hfs_{nm}'] for r in rows4]):+10.3f}")

print("\nSinkhorn win-rate:")
for key, label, fn, _ in METRICS:
    ws = [np.mean([fn(r[f'{key}_sink']) < fn(r[f'{key}_{nm}']) for r in rows4]) for nm in ['lin','ratio_free','ratio_prior']]
    print(f"  {label:22} vs lin {ws[0]*100:3.0f}%   vs direct {ws[1]*100:3.0f}%   vs direct+TV {ws[2]*100:3.0f}%")

In [ ]:
# ============================================================
# PRE-REGISTRATION: judge with your eyes BEFORE seeing any metric.
# Look at each row. Which midpoint is the better texture?
# Be honest — mark 'lin' where linear genuinely looks better.
# ============================================================
JUDGE_IDX = list(range(10))          # first 10 pairs of PAIRS_BIG

fig, axes = plt.subplots(len(JUDGE_IDX), 4, figsize=(11, 2.7*len(JUDGE_IDX)))
for row, i in enumerate(JUDGE_IDX):
    pa, pb, ca, cb, _ = PAIRS_BIG[i]
    d = build_d(pa, pb)
    cols = [(f'A: {ca}', d["ia"]),
            ('mid-1', interpolate_sinkhorn_pixel(d, 0.5)),   # deliberately unlabeled
            ('mid-2', interpolate_linear_pixel(d, 0.5)),
            (f'B: {cb}', d["ib"])]
    for j,(nm,t) in enumerate(cols):
        axes[row,j].imshow(np.clip(tensor_to_image(t),0,1)); axes[row,j].axis('off')
        if row==0: axes[row,j].set_title(nm, fontsize=11)
    axes[row,0].text(-0.1, 0.5, f"#{i}", transform=axes[row,0].transAxes,
                     fontsize=11, va='center', ha='right')
plt.suptitle("mid-1 = Sinkhorn, mid-2 = linear   (which is the better TEXTURE?)", fontsize=11)
plt.tight_layout(); plt.show()

print("\nFill this in by eye, THEN run the next cell:")
print("MY_JUDGMENT = {  # pair index -> 'sink' | 'lin' | 'tie'")
for i in JUDGE_IDX: print(f"    {i}: '',")
print("}")

In [ ]:
MY_JUDGMENT = {   # pair index -> 'sink' | 'lin' | 'tie'
    0: 'tie',    # stripes — both messy
    1: 'lin',    # fur — linear keeps more fibrous detail, sinkhorn washes out
    2: 'sink',   # honeycomb — cell structure stays coherent
    3: 'tie',    # grass/ornate — hard to call
    4: 'tie',    # rock strata — hard to call
    5: 'sink',   # foam/bubbles — weak, but linear has a seam artifact
    6: 'sink',   # chevron — linear GHOSTS badly, sinkhorn crisp
    7: 'sink',   # cracked earth — linear's cracks look doubled
    8: 'sink',   # mesh — linear has a white blob artifact
    9: 'tie',    # fish/grid — genuinely unclear
}
assert all(v in ('sink','lin','tie') for v in MY_JUDGMENT.values()), "fill in all 10 first"

# ============================================================
# LIKELIHOOD METRICS: is this midpoint a plausible sample from the texture distribution?
# ============================================================
from sklearn.covariance import LedoitWolf

# ---- build the reference distribution from REAL DTD textures (big sample) ----
print("fitting texture distribution...")
ref_feats = []
for c in rng.sample(ALL_CATS, min(40, len(ALL_CATS))):
    for p in sorted(glob.glob(os.path.join(DTD_ROOT, c, '*.jpg')))[:10]:
        img = load_texture(p)
        with torch.no_grad():
            f = vgg_dist.extract(img)[2]                 # mid-level VGG layer
            ref_feats.append(f.mean((2,3)).squeeze(0).cpu().numpy())   # pooled descriptor
X = np.stack(ref_feats)
print(f"  fitted on {len(X)} real textures, dim={X.shape[1]}")

lw = LedoitWolf().fit(X)                                  # shrinkage -> stable in high dim
mu = X.mean(0)

def gauss_loglik(img):
    """log p(x) under a Gaussian fit to real DTD features. HIGHER = more plausible."""
    with torch.no_grad():
        f = vgg_dist.extract(img)[2].mean((2,3)).squeeze(0).cpu().numpy()[None]
    return float(lw.score(f))                             # avg log-likelihood

# ---- Gram-space likelihood: sensitive to TEXTURE STRUCTURE, not just colour stats ----
def gram_vec(img):
    with torch.no_grad():
        f = vgg_dist.extract(img)[2]
        b,c,h,w = f.shape; x = f.reshape(c, h*w)
        G = (x @ x.t()) / (c*h*w)
        return G[np.triu_indices(c, 1)[0][:2000], np.triu_indices(c, 1)[1][:2000]].cpu().numpy()

print("fitting Gram-space distribution...")
Xg = np.stack([gram_vec(load_texture(p))
               for c in rng.sample(ALL_CATS, 30)
               for p in sorted(glob.glob(os.path.join(DTD_ROOT, c, '*.jpg')))[:6]])
lwg = LedoitWolf().fit(Xg)
def gram_loglik(img): return float(lwg.score(gram_vec(img)[None]))
print(f"  fitted on {len(Xg)} textures\n")

# ---- SANITY CHECK: does log-lik correctly rank the KNOWN failure modes? ----
print("VALIDATION — the metric must rank known-bad images as LOW likelihood:")
pa, pb, ca, cb, _ = PAIRS_BIG[0]
d = build_d(pa, pb)
img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
img_b = decode_interp(d["blob_b_gen"], d["bf_b"], d["si_b"])
d_ab = as_float(vgg_dist.distance(img_a, img_b))
probes = {
    'REAL texture A':      d["ia"],
    'Sinkhorn mid':        interpolate_sinkhorn_pixel(d, 0.5),
    'linear mid':          interpolate_linear_pixel(d, 0.5),
    'direct static (BAD)': direct_ratio_opt(d, img_a, img_b, d_ab, init_from_linear=False),
    'direct blur (BAD)':   direct_ratio_opt(d, img_a, img_b, d_ab, init_from_linear=True, tv_w=4.0, prox_w=0.1),
}
for nm, im in probes.items():
    print(f"  {nm:22} feat-loglik {gauss_loglik(im):9.1f}   gram-loglik {gram_loglik(im):9.1f}")
print("\n  ^ if the two BAD rows aren't clearly lowest, the metric is invalid. STOP if so.\n")

# ---- run on the judged pairs, then the full set ----
rows5 = []
for i in range(min(100, len(PAIRS_BIG))):
    pa, pb, ca, cb, _ = PAIRS_BIG[i]
    try:
        d = build_d(pa, pb)
        r = {'i': i, 'pair': f"{ca}/{cb}"}
        for nm, mid in [('sink', interpolate_sinkhorn_pixel(d,0.5)),
                        ('lin',  interpolate_linear_pixel(d,0.5))]:
            r[f'll_{nm}']   = gauss_loglik(mid)
            r[f'gll_{nm}']  = gram_loglik(mid)
        rows5.append(r)
    except Exception: continue
    if i % 25 == 0: print(f"  {i}/100")

print(f"\n=== LOG-LIKELIHOOD (higher = more plausible texture), n={len(rows5)} ===")
for key, label in [('ll','feature-space log p'), ('gll','Gram-space log p')]:
    s = np.mean([r[f'{key}_sink'] for r in rows5]); l = np.mean([r[f'{key}_lin'] for r in rows5])
    w = np.mean([r[f'{key}_sink'] > r[f'{key}_lin'] for r in rows5])
    print(f"  {label:22} sink {s:9.1f}   linear {l:9.1f}   sink wins {w*100:.0f}%")

# ---- THE TEST THAT MATTERS: does the metric agree with YOUR EYE? ----
judged = [r for r in rows5 if r['i'] in MY_JUDGMENT and MY_JUDGMENT[r['i']] != 'tie']
print(f"\n=== AGREEMENT WITH YOUR PRE-REGISTERED JUDGMENT (n={len(judged)}) ===")
for key, label in [('ll','feature log p'), ('gll','Gram log p')]:
    agree = np.mean([(r[f'{key}_sink'] > r[f'{key}_lin']) == (MY_JUDGMENT[r['i']] == 'sink')
                     for r in judged])
    print(f"  {label:16} agrees with your eye {agree*100:.0f}% of the time")
print("\n  >70% agreement -> the metric captures what you see; its verdict on all 200 is meaningful.")
print("  ~50%           -> it's not measuring what your eye measures. Don't use it.")

In [ ]:
# TYPICALITY, not density: how far is log p(x) from what REAL textures score?
real_lls = [gauss_loglik(load_texture(p))
            for c in rng.sample(ALL_CATS, 30)
            for p in sorted(glob.glob(os.path.join(DTD_ROOT, c, '*.jpg')))[:5]]
mu_ll, sd_ll = np.mean(real_lls), np.std(real_lls)
print(f"real textures: log p = {mu_ll:.1f} ± {sd_ll:.1f}")

def typicality(img):
    """|z-score| of log p vs real textures. 0 = scores like a real texture."""
    return abs(gauss_loglik(img) - mu_ll) / (sd_ll + 1e-9)

# re-run the SAME validation. it must rank blur and static as atypical.
for nm, im in probes.items():
    print(f"  {nm:22} typicality {typicality(im):6.2f}   (lower = more texture-like)")

In [ ]:
import os, glob, re, json, random
from PIL import Image

# --- TextureADE (from yesterday) ---
TADE = '/content/drive/MyDrive/DATA_ROOT/TextureADE'
assert os.path.isdir(os.path.join(TADE, 'images')), f"not found at {TADE}"

tade_ids = sorted({os.path.basename(p)[:-4] for p in glob.glob(os.path.join(TADE,'images','*.jpg'))})
meta = json.load(open(os.path.join(TADE,'metadata.json')))
DESC = {m['id']: [(t['description'], os.path.basename(t['mask_path'])) for t in m['textures']] for m in meta}

def region_crop(img_id, region_k, out_size=256):
    """Crop the tight bbox of region k, resized to a square patch."""
    img = Image.open(os.path.join(TADE,'images',f'{img_id}.jpg')).convert('RGB')
    mp  = os.path.join(TADE,'textures_mask',f'{img_id}_mask_{region_k}.png')
    m   = np.asarray(Image.open(mp).convert('L').resize(img.size, Image.NEAREST)) > 127
    ys, xs = np.where(m)
    if len(ys) < 50: return None
    y0,y1,x0,x1 = ys.min(), ys.max(), xs.min(), xs.max()
    if (y1-y0) < 32 or (x1-x0) < 32: return None
    return img.crop((x0,y0,x1,y1)).resize((out_size,out_size), Image.BILINEAR)

def full_image(img_id, out_size=256):
    return Image.open(os.path.join(TADE,'images',f'{img_id}.jpg')).convert('RGB').resize((out_size,out_size))

# --- classify each region by how "semantic" vs "texture-like" it is ---
STUFF   = ['sky','grass','tree','foliage','water','sea','sand','earth','rock','road',
           'mountain','wall','floor','snow','field','river','stone','dirt','cloud']
OBJECT  = ['car','person','building','house','window','door','chair','table','sign',
           'boat','bus','lamp','fence','pole','bed','sofa','plant pot']

def region_kind(desc):
    d = desc.lower()
    if any(w in d for w in OBJECT): return 'object'
    if any(w in d for w in STUFF):  return 'stuff'
    return 'other'

pool = {'stuff': [], 'object': []}
for iid in tade_ids:
    for k,(desc,_) in enumerate(DESC[iid]):
        kind = region_kind(desc)
        if kind in pool: pool[kind].append((iid, k, desc))
print(f"TextureADE regions: stuff={len(pool['stuff'])}  object={len(pool['object'])}")

# --- DTD helpers (reuse from earlier cells) ---
def dtd_pil(cat, seed):
    ps = sorted(glob.glob(os.path.join(DTD_ROOT, cat, '*.jpg')))
    return Image.open(random.Random(seed).choice(ps)).convert('RGB').resize((256,256))

DTD_TEXTURE = ['banded','dotted','striped','chequered','honeycombed','woven','cracked','scaly']

In [ ]:
rnd = random.Random(7)
N_PER_LEVEL = 8

def sample_level(level):
    """returns list of (pil_a, pil_b, label_a, label_b)"""
    out = []
    for i in range(N_PER_LEVEL):
        if level == 0:      # pure texture <-> pure texture  (BASELINE)
            ca, cb = rnd.sample(DTD_TEXTURE, 2)
            out.append((dtd_pil(ca, i), dtd_pil(cb, i+50), ca, cb))

        elif level == 1:    # texture <-> texture-like stuff region (CROP)
            ca = rnd.choice(DTD_TEXTURE)
            iid, k, desc = rnd.choice(pool['stuff'])
            c = region_crop(iid, k)
            if c: out.append((dtd_pil(ca, i), c, ca, desc.split(',')[0][:28]))

        elif level == 2:    # stuff <-> stuff, incompatible semantics (CROP)
            (i1,k1,d1), (i2,k2,d2) = rnd.sample(pool['stuff'], 2)
            c1, c2 = region_crop(i1,k1), region_crop(i2,k2)
            if c1 and c2: out.append((c1, c2, d1.split(',')[0][:28], d2.split(',')[0][:28]))

        elif level == 3:    # texture <-> OBJECT region (semantic structure present)
            ca = rnd.choice(DTD_TEXTURE)
            iid, k, desc = rnd.choice(pool['object'])
            c = region_crop(iid, k)
            if c: out.append((dtd_pil(ca, i), c, ca, desc.split(',')[0][:28]))

        elif level == 4:    # FULL SCENE <-> FULL SCENE (no crop; real semantic images)
            i1, i2 = rnd.sample(tade_ids, 2)
            out.append((full_image(i1), full_image(i2), 'scene', 'scene'))
    return out

LEVELS = {
    0: 'texture <-> texture (DTD baseline)',
    1: 'texture <-> stuff-crop',
    2: 'stuff-crop <-> stuff-crop',
    3: 'texture <-> OBJECT-crop',
    4: 'full scene <-> full scene',
}
PAIRSETS = {L: sample_level(L) for L in LEVELS}
for L, ps in PAIRSETS.items():
    print(f"L{L}  {LEVELS[L]:38} {len(ps)} pairs")

In [ ]:
import tempfile

def build_d_from_pil(pil_a, pil_b):
    """CNT pipeline expects paths -> write temp files."""
    fa = tempfile.NamedTemporaryFile(suffix='.jpg', delete=False); pil_a.save(fa.name)
    fb = tempfile.NamedTemporaryFile(suffix='.jpg', delete=False); pil_b.save(fb.name)
    return build_d(fa.name, fb.name)

results = {}
for L, ps in PAIRSETS.items():
    rows = []
    for pil_a, pil_b, la, lb in ps:
        try:
            d = build_d_from_pil(pil_a, pil_b)
            img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
            img_b = decode_interp(d["blob_b_gen"], d["bf_b"], d["si_b"])
            d_ab  = as_float(vgg_dist.distance(img_a, img_b))
            r = {'la': la, 'lb': lb}
            for nm, fn in [('sink', interpolate_sinkhorn_pixel), ('lin', interpolate_linear_pixel)]:
                mid = fn(d, 0.5)
                r[f'ratio_{nm}'] = as_float(vgg_dist.distance(img_a, mid)) / max(d_ab, 1e-9)
                r[f'hf_{nm}']    = hf_signed(mid, img_a, img_b)
            rows.append(r)
        except Exception as e:
            print(f"  L{L} skip: {type(e).__name__}")
    results[L] = rows
    if rows:
        s = np.mean([abs(x['ratio_sink']-0.5) for x in rows])
        l = np.mean([abs(x['ratio_lin'] -0.5) for x in rows])
        w = np.mean([abs(x['ratio_sink']-0.5) < abs(x['ratio_lin']-0.5) for x in rows])
        print(f"L{L} {LEVELS[L]:38} |ratio-0.5|: sink {s:.3f}  lin {l:.3f}  sink wins {w*100:3.0f}%  (n={len(rows)})")

print("\nKEY QUESTION: does |ratio-0.5| stay LOW even at L3/L4, where the images are")
print("semantically absurd? If yes -> perceptual distance certifies NOTHING about validity.")

In [ ]:
for L in [1, 3, 4]:
    ps = PAIRSETS[L][:4]
    fig, axes = plt.subplots(len(ps), 4, figsize=(11, 2.8*len(ps)))
    for i,(pil_a, pil_b, la, lb) in enumerate(ps):
        try:
            d = build_d_from_pil(pil_a, pil_b)
            img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
            img_b = decode_interp(d["blob_b_gen"], d["bf_b"], d["si_b"])
            d_ab  = as_float(vgg_dist.distance(img_a, img_b))
            ms = interpolate_sinkhorn_pixel(d, 0.5)
            ml = interpolate_linear_pixel(d, 0.5)
            rs = as_float(vgg_dist.distance(img_a, ms))/max(d_ab,1e-9)
            rl = as_float(vgg_dist.distance(img_a, ml))/max(d_ab,1e-9)
            cols = [(la[:22], np.asarray(pil_a)/255.),
                    (f'Sinkhorn (ratio {rs:.2f})', np.clip(tensor_to_image(ms),0,1)),
                    (f'linear (ratio {rl:.2f})',   np.clip(tensor_to_image(ml),0,1)),
                    (lb[:22], np.asarray(pil_b)/255.)]
            for j,(t, im) in enumerate(cols):
                axes[i,j].imshow(im); axes[i,j].axis('off'); axes[i,j].set_title(t, fontsize=8)
        except Exception: pass
    fig.suptitle(f"L{L}: {LEVELS[L]}", fontsize=11)
    plt.tight_layout()
    plt.savefig(f'/content/drive/MyDrive/DATA_ROOT/break_L{L}.png', dpi=130, bbox_inches='tight')
    plt.show()

In [ ]:
# ============================================================
# BREAK IT: FACES — the regime where correspondence EXISTS
# Prediction: linear should WIN here (spatial position = correspondence),
# because Sinkhorn transports by feature similarity, ignoring semantic parts.
# ============================================================
!pip install -q datasets
from datasets import load_dataset
from PIL import Image
import numpy as np

# LFW: small, aligned faces, loads fast. (fallback below if it stalls)
try:
    faces_ds = load_dataset("vilsonrodrigues/lfw", split="train[:60]")
    face_pils = [ex['image'].convert('RGB').resize((256,256)) for ex in faces_ds]
    print(f"loaded {len(face_pils)} LFW faces")
except Exception as e:
    print("LFW failed:", type(e).__name__, "— falling back to DTD 'freckled' (contains faces)")
    fp = sorted(glob.glob(os.path.join(DTD_ROOT, 'freckled', '*.jpg')))
    face_pils = [Image.open(p).convert('RGB').resize((256,256)) for p in fp[:40]]
    print(f"loaded {len(face_pils)} freckled/face images")

rnd = random.Random(11)

LEVELS_HARD = {
    5: 'face <-> face  (ALIGNED — correspondence EXISTS)',
    6: 'face <-> texture  (extreme: does a midpoint exist?)',
    7: 'face <-> full scene',
}

def sample_hard(level, n=10):
    out = []
    for i in range(n):
        if level == 5:
            a, b = rnd.sample(range(len(face_pils)), 2)
            out.append((face_pils[a], face_pils[b], 'face A', 'face B'))
        elif level == 6:
            a = rnd.choice(face_pils)
            c = rnd.choice(DTD_TEXTURE)
            out.append((a, dtd_pil(c, i), 'face', c))
        elif level == 7:
            a = rnd.choice(face_pils)
            iid = rnd.choice(tade_ids)
            out.append((a, full_image(iid), 'face', 'scene'))
    return out

HARD = {L: sample_hard(L) for L in LEVELS_HARD}

for L, ps in HARD.items():
    rows = []
    for pil_a, pil_b, la, lb in ps:
        try:
            d = build_d_from_pil(pil_a, pil_b)
            img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
            img_b = decode_interp(d["blob_b_gen"], d["bf_b"], d["si_b"])
            d_ab  = as_float(vgg_dist.distance(img_a, img_b))
            r = {}
            for nm, fn in [('sink', interpolate_sinkhorn_pixel), ('lin', interpolate_linear_pixel)]:
                mid = fn(d, 0.5)
                r[f'ratio_{nm}'] = as_float(vgg_dist.distance(img_a, mid)) / max(d_ab,1e-9)
                r[f'hf_{nm}']    = hf_signed(mid, img_a, img_b)
            rows.append(r)
        except Exception as e:
            print(f"  L{L} skip: {type(e).__name__}")
    if rows:
        s = np.mean([abs(x['ratio_sink']-0.5) for x in rows])
        l = np.mean([abs(x['ratio_lin']-0.5) for x in rows])
        w = np.mean([abs(x['ratio_sink']-0.5) < abs(x['ratio_lin']-0.5) for x in rows])
        print(f"L{L} {LEVELS_HARD[L]:42} |ratio-0.5|: sink {s:.3f}  lin {l:.3f}  sink wins {w*100:3.0f}%  (n={len(rows)})")

# ---- the figure: DOES SINKHORN SCRAMBLE THE FACE? ----
for L in [5, 6]:
    ps = HARD[L][:4]
    fig, axes = plt.subplots(len(ps), 4, figsize=(11, 2.8*len(ps)))
    for i,(pil_a, pil_b, la, lb) in enumerate(ps):
        try:
            d = build_d_from_pil(pil_a, pil_b)
            img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
            img_b = decode_interp(d["blob_b_gen"], d["bf_b"], d["si_b"])
            d_ab  = as_float(vgg_dist.distance(img_a, img_b))
            ms, ml = interpolate_sinkhorn_pixel(d,0.5), interpolate_linear_pixel(d,0.5)
            rs = as_float(vgg_dist.distance(img_a, ms))/max(d_ab,1e-9)
            rl = as_float(vgg_dist.distance(img_a, ml))/max(d_ab,1e-9)
            cols = [(la, np.asarray(pil_a)/255.),
                    (f'Sinkhorn ({rs:.2f})', np.clip(tensor_to_image(ms),0,1)),
                    (f'linear ({rl:.2f})',   np.clip(tensor_to_image(ml),0,1)),
                    (lb, np.asarray(pil_b)/255.)]
            for j,(t,im) in enumerate(cols):
                axes[i,j].imshow(im); axes[i,j].axis('off'); axes[i,j].set_title(t, fontsize=9)
        except Exception: pass
    fig.suptitle(f"L{L}: {LEVELS_HARD[L]}", fontsize=11)
    plt.tight_layout()
    plt.savefig(f'/content/drive/MyDrive/DATA_ROOT/break_L{L}.png', dpi=130, bbox_inches='tight')
    plt.show()

In [ ]:
# Diagnose L5 specifically
print("face_pils loaded:", len(face_pils) if 'face_pils' in dir() else "NOT DEFINED")
print("HARD[5] pairs:", len(HARD[5]) if 'HARD' in dir() and 5 in HARD else "NOT BUILT")

if 'HARD' in dir() and 5 in HARD and len(HARD[5]) > 0:
    pil_a, pil_b, la, lb = HARD[5][0]
    print("pair labels:", la, lb, "| sizes:", pil_a.size, pil_b.size, "| modes:", pil_a.mode, pil_b.mode)
    # show them so we know WHAT we're interpolating
    fig, ax = plt.subplots(1, 2, figsize=(6,3))
    ax[0].imshow(pil_a); ax[0].axis('off'); ax[0].set_title('A')
    ax[1].imshow(pil_b); ax[1].axis('off'); ax[1].set_title('B')
    plt.show()
    # run ONE pair with the real traceback
    import traceback
    try:
        d = build_d_from_pil(pil_a, pil_b)
        img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
        img_b = decode_interp(d["blob_b_gen"], d["bf_b"], d["si_b"])
        d_ab  = as_float(vgg_dist.distance(img_a, img_b))
        ms = interpolate_sinkhorn_pixel(d, 0.5)
        ml = interpolate_linear_pixel(d, 0.5)
        rs = as_float(vgg_dist.distance(img_a, ms))/max(d_ab,1e-9)
        rl = as_float(vgg_dist.distance(img_a, ml))/max(d_ab,1e-9)
        print(f"WORKS: sink ratio {rs:.3f}   linear ratio {rl:.3f}")
        fig, ax = plt.subplots(1,4, figsize=(12,3))
        for a,(t,im) in zip(ax, [('A', np.asarray(pil_a)/255.),
                                 (f'Sinkhorn {rs:.2f}', np.clip(tensor_to_image(ms),0,1)),
                                 (f'linear {rl:.2f}',   np.clip(tensor_to_image(ml),0,1)),
                                 ('B', np.asarray(pil_b)/255.)]):
            a.imshow(im); a.axis('off'); a.set_title(t)
        plt.show()
    except Exception:
        traceback.print_exc()

In [ ]:
# Option A: retry LFW via sklearn (bundled fetcher, different CDN, usually reliable)
from sklearn.datasets import fetch_lfw_people
lfw = fetch_lfw_people(min_faces_per_person=20, resize=1.0, color=True, funneled=True)
imgs = lfw.images  # (N, H, W, 3) float 0-1, ALIGNED and funneled
face_pils = [Image.fromarray((imgs[i]*255).astype(np.uint8)).resize((256,256))
             for i in range(min(60, len(imgs)))]
print(f"loaded {len(face_pils)} ALIGNED LFW faces, shape {imgs.shape}")

# sanity: they should all be centered frontal faces
fig, ax = plt.subplots(1, 5, figsize=(12,2.6))
for a, p in zip(ax, face_pils[:5]):
    a.imshow(p); a.axis('off')
plt.suptitle("if these are aligned frontal faces, L5 is now a real test")
plt.show()

In [ ]:
# Rebuild L5 with the ALIGNED faces and run it
HARD[5] = sample_hard(5, n=10)

rows = []
for pil_a, pil_b, la, lb in HARD[5]:
    try:
        d = build_d_from_pil(pil_a, pil_b)
        img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
        img_b = decode_interp(d["blob_b_gen"], d["bf_b"], d["si_b"])
        d_ab  = as_float(vgg_dist.distance(img_a, img_b))
        r = {}
        for nm, fn in [('sink', interpolate_sinkhorn_pixel), ('lin', interpolate_linear_pixel)]:
            mid = fn(d, 0.5)
            r[f'ratio_{nm}'] = as_float(vgg_dist.distance(img_a, mid)) / max(d_ab, 1e-9)
        rows.append(r)
    except Exception as e:
        print("skip:", type(e).__name__)

s = np.mean([abs(x['ratio_sink']-0.5) for x in rows])
l = np.mean([abs(x['ratio_lin'] -0.5) for x in rows])
w = np.mean([abs(x['ratio_sink']-0.5) < abs(x['ratio_lin']-0.5) for x in rows])
print(f"\nL5 ALIGNED FACES (n={len(rows)}):  |ratio-0.5|  sink {s:.3f}   linear {l:.3f}   sink wins {w*100:.0f}%")
print("PREDICTION: linear should win here — spatial position IS the correspondence.\n")

# THE FIGURE — does Sinkhorn keep the eyes where they belong?
ps = HARD[5][:5]
fig, axes = plt.subplots(len(ps), 4, figsize=(11, 2.8*len(ps)))
for i,(pil_a, pil_b, la, lb) in enumerate(ps):
    d = build_d_from_pil(pil_a, pil_b)
    img_a = decode_interp(d["blob_a_gen"], d["bf_a"], d["si_a"])
    img_b = decode_interp(d["blob_b_gen"], d["bf_b"], d["si_b"])
    d_ab  = as_float(vgg_dist.distance(img_a, img_b))
    ms, ml = interpolate_sinkhorn_pixel(d,0.5), interpolate_linear_pixel(d,0.5)
    rs = as_float(vgg_dist.distance(img_a, ms))/max(d_ab,1e-9)
    rl = as_float(vgg_dist.distance(img_a, ml))/max(d_ab,1e-9)
    for j,(t,im) in enumerate([('face A', np.asarray(pil_a)/255.),
                               (f'Sinkhorn {rs:.2f}', np.clip(tensor_to_image(ms),0,1)),
                               (f'linear {rl:.2f}',   np.clip(tensor_to_image(ml),0,1)),
                               ('face B', np.asarray(pil_b)/255.)]):
        axes[i,j].imshow(im); axes[i,j].axis('off'); axes[i,j].set_title(t, fontsize=9)
plt.suptitle("L5: ALIGNED faces — correspondence EXISTS. Does OT scramble it?", fontsize=11)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/DATA_ROOT/break_L5_faces.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
import pickle, json, os, time

SAVE = '/content/drive/MyDrive/DATA_ROOT/session_2026_07_14'
os.makedirs(SAVE, exist_ok=True)

bundle = {
    # --- Item 1: LPIPS-direct baseline (CLOSED) ---
    'direct_200':        rows2  if 'rows2' in dir() else None,   # n=200, ratio + manifold, within/cross
    'direct_fair_obj':   rows3  if 'rows3' in dir() else None,   # metric-optimal baseline, n=100
    'benchmark_v2':      rows4  if 'rows4' in dir() else None,   # gram / HF / signed-HF, n=100
    'likelihood_failed': rows5  if 'rows5' in dir() else None,   # INVALID — kept for the record
    # --- Item 4: break-it gradient ---
    'break_levels':      results if 'results' in dir() else None,# L0-L4
    'break_hard':        HARD   if 'HARD'  in dir() else None,   # L5-L7 pair sets
    'my_judgment':       MY_JUDGMENT if 'MY_JUDGMENT' in dir() else None,
}
pickle.dump(bundle, open(os.path.join(SAVE,'session_bundle.pkl'),'wb'))

config = {
    'saved': time.strftime('%Y-%m-%d %H:%M'),
    'headline_results': {
        'core_claim': 'Sinkhorn ratio 0.584 vs pixel-linear 0.715 (n=200, within+cross) — replicates the 7000-pair slide',
        'direct_baseline': 'metric-optimal direct opt hits ratio 0.500 (perfect) while producing STATIC. '
                           'signed-HF exposes it: static +1.86, blur -2.92, sinkhorn +0.34',
        'break_gradient_sink_winrate': {'L0_texture':'75%', 'L1_tex-stuff':'100%', 'L2_stuff-stuff':'88%',
                                        'L3_tex-object':'100%', 'L4_full_scene':'100%', 'L5_aligned_faces':'70%'},
        'L5_KEY': 'aligned faces: sink |ratio-0.5| = 0.077 (near-ideal) on images that are DESTROYED faces. '
                  'Strongest evidence that perceptual ratio certifies NOTHING about validity.',
    },
    'metrics_tried_and_verdict': {
        'vgg_ratio':        'VALID for placement, but SATURABLE (direct opt hits 0.500)',
        'balance_ratio':    'REJECTED — rewards off-manifold equidistance',
        'gram_ratio':       'no separation between methods',
        'gram_manifold':    'REJECTED — rewards blur',
        'signed_HF':        'VALID — correctly IDs static (+) and blur (-). KEEP.',
        'gaussian_loglik':  'REJECTED — INVERTED (real texture scored lowest, blur highest)',
        'typicality':       'REJECTED — failed same validation',
    },
    'theory_update': 'Correspondence law as stated is NOT supported: OT wins in all 7 regimes incl. aligned faces. '
                     'Revised hypothesis: CNT texton representation destroys spatial correspondence for ANY input, '
                     'so every image is forced into the no-correspondence regime. TESTABLE: compare vs raw-pixel-space '
                     'interpolation where alignment survives.',
    'items': {
        '1_lpips_direct':      'DONE (n=200 + fair-objective + signed-HF)',
        '2_representation':    'PARTIAL — have CNT+33/+27, SAM+13, DINO+6; inversion sweep is only n=10. '
                               'TOMORROW: bump cnt_sam_dino_comapre cell5 to VGG_PAIRS=sample_pairs[:40]',
        '3_cross_observer':    'BLOCKED — only the dataset is available, no runnable model.',
        '4_break_it':          'DONE — and it did NOT break; OT advantage GROWS with difficulty',
    },
    'next_steps': [
        'Run cnt_sam_dino_comapre: N_PAIRS=100 (cell3) and VGG_PAIRS=sample_pairs[:40] (cell5) — closes item 2',
        'Scale break-gradient to n=40-50 per level (currently n=8-10)',
        'Test the CNT-destroys-correspondence hypothesis: raw-pixel-space interpolation on aligned faces',
    ],
}
json.dump(config, open(os.path.join(SAVE,'config.json'),'w'), indent=2)

restore = '''# RUN TOMORROW. Kernel is dead; re-run setup cells first, THEN this.
# 1. Mount drive, load CNT model, vgg_dist, interpolators, build_d, direct_ratio_opt, hf_signed
# 2. Then:
import pickle, json
S = "/content/drive/MyDrive/DATA_ROOT/session_2026_07_14"
b   = pickle.load(open(S+"/session_bundle.pkl","rb"))
cfg = json.load(open(S+"/config.json"))
rows2, rows3, rows4 = b["direct_200"], b["direct_fair_obj"], b["benchmark_v2"]
results, HARD = b["break_levels"], b["break_hard"]
print(json.dumps(cfg["headline_results"], indent=2))
print("\nNext steps:", *cfg["next_steps"], sep="\\n  - ")
'''
open(os.path.join(SAVE,'restore.py'),'w').write(restore)

print("saved to", SAVE)
for f in sorted(os.listdir(SAVE)):
    print(f"   {f}  ({os.path.getsize(os.path.join(SAVE,f))} b)")
print("\nfigures already in Drive: break_L*.png, direct_*.png, fig*.png")